# Two-channel waveform coincidence analysis

This notebook loads two-channel waveform events (either single files with two channels per sample, or paired files named with `_ch1`/`_ch2` conventions), computes per-event amplitudes and peak times, then plots both channels per event and records deltaT and amplitudes into a summary CSV.

Outputs: per-event PNGs in `Data/Analysis_results/two_channel/` and a summary CSV `two_channel_summary.csv`.

In [1]:
# Imports and configuration
import os, re, csv, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks

# try to import lecroyparser for .trc reading; fall back to text heuristics if unavailable
try:
    import lecroyparser
    LECROY_AVAILABLE = True
except Exception:
    LECROY_AVAILABLE = False

# Adjust these to your workspace layout
WORKDIR = Path('/Users/ukose/sw/Work/Neutron3D')
WAVEFORM_DIR = WORKDIR / 'Data' / 'Waveforms' / 'Sample2' / 'AmBe_thermal_n_coincidence_1750V'
OUT_DIR = WORKDIR / 'Data' / 'Analysis_results' / 'two_channel'
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Expected sample rate or time units: if the file includes time values, the loader will use them.
DEFAULT_POLARITY = -1.0  # PMT pulses are negative; multiply signal by this to make peaks positive when extracting amplitude
PRINT_EVERY = 50

PermissionError: [Errno 13] Permission denied: '/Users/ukose'

In [ ]:
def find_two_channel_pairs(folder, pattern_ch1='C1_', pattern_ch2='C2_'):
    """Find pairs of files in a folder where one filename contains pattern_ch1 and the partner contains pattern_ch2 with the same stem.
    Returns list of (path_ch1, path_ch2).
    """
    p = Path(folder)
    files = sorted([f for f in p.glob('**/*') if f.is_file()])
    by_stem = {}
    pairs = []
    for f in files:
        name = f.name
        if pattern_ch1 in name or pattern_ch2 in name:
            # derive base by removing patterns and extension
            base = name.replace(pattern_ch1, '').replace(pattern_ch2, '')
            # remove final extension (e.g. .trc) to match stems
            base = re.sub(r'\.[^.]+$', '', base)
            if base not in by_stem: by_stem[base] = {}
            if pattern_ch1 in name: by_stem[base]['ch1'] = f
            if pattern_ch2 in name: by_stem[base]['ch2'] = f
    for base, d in by_stem.items():
        if 'ch1' in d and 'ch2' in d:
            pairs.append((d['ch1'], d['ch2']))
    return pairs


def find_files_with_two_channels(folder):
    """Find files that appear to contain two channels per sample (e.g., three columns: time, ch1, ch2).
    Returns list of file paths.
    """
    p = Path(folder)
    candidates = []
    for f in p.glob('**/*'):
        if not f.is_file(): continue
        # quick sniff: check first non-empty non-comment line for 2 or 3 numeric columns
        try:
            with f.open('r') as fh:
                for _ in range(50):
                    line = fh.readline()
                    if not line: break
                    line = line.strip()
                    if not line or line.startswith('#'): continue
                    parts = re.split('[,\t ]+', line)
                    nums = 0
                    for pstr in parts[:4]:
                        try:
                            float(pstr)
                            nums += 1
                        except Exception:
                            pass
                    if nums >= 3:
                        candidates.append(f)
                    break
        except Exception:
            continue
    return candidates

In [ ]:
def load_two_channel_file(path):
    """Load a file that contains time,ch1,ch2 or ch1,ch2 (no time), or a LeCroy .trc file.
    Returns (t, ch1, ch2, meta) where t may be None if absent. For single-channel .trc files we return ch1 filled and ch2 empty.
    """
    path = Path(path)
    # handle LeCroy .trc binary files if lecroyparser is available
    if path.suffix.lower() == '.trc':
        if LECROY_AVAILABLE:
            try:
                scope = lecroyparser.ScopeData(str(path), parseAll=False)
                # ScopeData exposes x (time in seconds) and y (voltage samples) for single-channel captures
                t = np.asarray(getattr(scope, 'x', []), dtype=float)
                y = np.asarray(getattr(scope, 'y', []), dtype=float)
                meta = {'path': str(path)}
                return (t if t.size else None, y, np.array([]), meta)
            except Exception as exc:
                # fall back to text parsing below if lecroyparser fails
                print(f"lecroyparser failed to read {path}: {exc}")
        else:
            print(f"Warning: .trc file {path} found but lecroyparser is not installed; trying text heuristic")
    # fallback: try to read as text with time,ch1,ch2 or ch1,ch2 columns
    t_list = []
    ch1_list = []
    ch2_list = []
    try:
        with path.open('r', errors='ignore') as fh:
            for line in fh:
                s = line.strip()
                if not s or s.startswith('#') or s.upper().startswith('LECROY') or s.upper().startswith('TIME'):
                    continue
                s2 = re.sub('[,\t]+', ' ', s)
                parts = s2.split()
                if len(parts) >= 3:
                    try:
                        t = float(parts[0])
                        a = float(parts[1])
                        b = float(parts[2])
                        t_list.append(t)
                        ch1_list.append(a)
                        ch2_list.append(b)
                    except Exception:
                        continue
                elif len(parts) == 2:
                    try:
                        a = float(parts[0])
                        b = float(parts[1])
                        ch1_list.append(a)
                        ch2_list.append(b)
                    except Exception:
                        continue
    except Exception:
        # if even text open fails, return empty arrays
        return None, np.array([]), np.array([]), {'path': str(path)}
    t = np.array(t_list) if t_list else None
    ch1 = np.array(ch1_list) if ch1_list else np.array([])
    ch2 = np.array(ch2_list) if ch2_list else np.array([])
    meta = {'path': str(path)}
    return t, ch1, ch2, meta

In [ ]:
def baseline_and_amplitude(t, sig, polarity=-1.0, baseline_samples=50, smooth=True):
    """Return baseline, baseline-subtracted signal, amplitude (peak), and peak time.
    polarity: multiply signal by polarity to make pulses positive if necessary.

    This function is defensive: if the input signal is empty or contains only NaNs,
    it returns NaN metrics instead of raising an exception.
    Uses baseline RMS to set a noise-aware prominence threshold so small pulses
    near the noise floor can be detected more robustly.
    """
    sig = np.asarray(sig, dtype=float)
    n = len(sig)
    if n == 0:
        return np.nan, sig, np.nan, np.nan
    bs = int(min(baseline_samples, n))
    baseline = np.nanmean(sig[:bs])
    sig_corr = sig - baseline
    # estimate noise from baseline region
    try:
        noise_rms = float(np.nanstd(sig[:bs])) if bs>1 else 0.0
    except Exception:
        noise_rms = 0.0
    if smooth and n >= 7:
        # small smoothing to help peak find (window 11 or smaller if short)
        w = min(11, n//2*2-1)
        try:
            sig_s = savgol_filter(sig_corr, w, 3)
        except Exception:
            sig_s = sig_corr
    else:
        sig_s = sig_corr
    # make pulse positive for detection
    sdet = polarity * sig_s
    # defensive checks: if sdet has no finite values, return NaNs
    if sdet.size == 0 or np.all(np.isnan(sdet)) or not np.isfinite(np.nanmax(sdet)):
        return baseline, sig_corr, np.nan, np.nan
    # compute adaptive prominence: combine a fraction of the peak with multiple of noise RMS
    maxv = np.nanmax(sdet)
    # at least 4*sigma noise or 5% of peak (whichever is larger) - avoids excessive false positives
    prominence_noise = 4.0 * noise_rms if np.isfinite(noise_rms) else 0.0
    prominence_peak = (maxv * 0.05) if (np.isfinite(maxv) and maxv > 0.0) else 0.0
    prominence = max(prominence_noise, prominence_peak)
    # ensure non-negative
    prominence = float(max(prominence, 0.0))
    peaks, props = find_peaks(sdet, prominence=prominence)
    if len(peaks) == 0:
        # fallback: use global max of sdet only if it's above a minimal noise threshold
        try:
            idx = int(np.nanargmax(sdet))
            # if the found max is below a conservative noise threshold, treat as no peak
            if np.isfinite(noise_rms) and sdet[idx] < 2.0 * noise_rms:
                return baseline, sig_corr, np.nan, np.nan
        except ValueError:
            return baseline, sig_corr, np.nan, np.nan
    else:
        # choose the peak with largest prominence when available
        if 'prominences' in props and len(props['prominences']) == len(peaks):
            idx = int(peaks[np.argmax(props['prominences'])])
        else:
            idx = int(peaks[0])
    amp = sdet[idx] if np.isfinite(sdet[idx]) else np.nan
    tpeak = None if t is None else (t[idx] if len(t) > idx else np.nan)
    return baseline, sig_corr, amp, tpeak

In [ ]:
def plot_two_channel_event(t, ch1, ch2, outpath, title=None, polarity=-1.0):
    plt.figure(figsize=(9,5))
    ax1 = plt.subplot(2,1,1)
    plt.plot(t if t is not None else np.arange(len(ch1)), ch1, label='ch1')
    b1, s1, a1, t1 = baseline_and_amplitude(t, ch1, polarity=polarity)
    plt.axhline(b1, color='gray', linestyle='--')
    if t1 is not None: plt.axvline(t1, color='C1')
    plt.ylabel('Ch1')
    plt.legend()
    ax2 = plt.subplot(2,1,2, sharex=ax1)
    plt.plot(t if t is not None else np.arange(len(ch2)), ch2, label='ch2')
    b2, s2, a2, t2 = baseline_and_amplitude(t, ch2, polarity=polarity)
    plt.axhline(b2, color='gray', linestyle='--')
    if t2 is not None: plt.axvline(t2, color='C1')
    plt.ylabel('Ch2')
    plt.legend()
    if title: plt.suptitle(title)
    # annotate deltaT and amplitudes
    if (t1 is not None) and (t2 is not None):
        dt = t2 - t1
        plt.gcf().text(0.7, 0.92, f'Δt={dt:.3e} s')
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(outpath, dpi=200)
    plt.close()
    return dict(amplitude_ch1=a1, tpeak_ch1=t1, amplitude_ch2=a2, tpeak_ch2=t2)

In [ ]:
# Main processing: try paired files first, then multi-channel single files
pairs = find_two_channel_pairs(WAVEFORM_DIR, pattern_ch1='C1_', pattern_ch2='C2_')
single_candidates = find_files_with_two_channels(WAVEFORM_DIR)
print(f'Found {len(pairs)} paired files and {len(single_candidates)} single two-channel candidates')
summary = []
# process pairs
for i,(p1,p2) in enumerate(pairs):
    t,ch1,ch2,meta = load_pair_files(p1,p2)
    stem = Path(p1).stem + '_' + Path(p2).stem
    outpng = OUT_DIR / f'{stem}_event.png'
    res = plot_two_channel_event(t,ch1,ch2,str(outpng), title=stem, polarity=DEFAULT_POLARITY)
    hv_match = re.search(r"_(\\d{3,4})V", p1.name)
    hv = int(hv_match.group(1)) if hv_match else None
    summary.append({'file1':str(p1),'file2':str(p2),'hv':hv, **res})
    if i%PRINT_EVERY==0: print(f'Processed pair {i}/{len(pairs)}')

# process single two-channel files
for j,f in enumerate(single_candidates):
    t,ch1,ch2,meta = load_two_channel_file(f)
    stem = Path(f).stem
    outpng = OUT_DIR / f'{stem}_event.png'
    res = plot_two_channel_event(t,ch1,ch2,str(outpng), title=stem, polarity=DEFAULT_POLARITY)
    hv_match = re.search(r"_(\\d{3,4})V", f.name)
    hv = int(hv_match.group(1)) if hv_match else None
    summary.append({'file1':str(f),'file2':'','hv':hv, **res})
    if j%PRINT_EVERY==0: print(f'Processed single {j}/{len(single_candidates)}')

# write summary CSV
outcsv = OUT_DIR / 'two_channel_summary.csv'
with outcsv.open('w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=['file1','file2','hv','amplitude_ch1','tpeak_ch1','amplitude_ch2','tpeak_ch2'])
    w.writeheader()
    for r in summary: w.writerow(r)

print('Done. Wrote', outcsv)

Found 811 paired files and 0 single two-channel candidates


NameError: name 'load_pair_files' is not defined

Notes:
- The loader heuristics try to detect files with three columns (time,ch1,ch2) or pairs of files matching patterns.
- `DEFAULT_POLARITY` is set to -1.0 (negative pulses). If your pulses are positive, set it to +1.0.
- The peak finder uses a prominence threshold; you may need to tune the `baseline_and_amplitude` function for your data (prominence fraction, baseline window).
- Tell me if you want amplitude integrated charge (area) instead of peak amplitude, or if you prefer CSV fields changed/extended.